# Example 7.6 / 1: Count Values with numberof

Companion notebook for **7.6 Counting Constraints** in the AMPL lab Documentation, section "Count Values with numberof". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Count Values with numberof

Five jobs go to three machines numbered 1 to 3. `MachineForJob[j]` holds the machine number of
job `j`, and `numberof m in (...)` counts how many jobs chose machine `m`. Lower-numbered machines
are faster, so heavier jobs should go there:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set JOBS;                                   # jobs to assign
param weight {JOBS} > 0;                    # importance of finishing the job early
param number_of_machines integer > 0;       # machines available
set MACHINES := 1..number_of_machines;      # machines numbered from 1, fastest first
param capacity {MACHINES} integer >= 0;     # most jobs a machine can take
var MachineForJob {JOBS} integer >= 1, <= number_of_machines;  # machine number assigned

minimize WeightedMachine: sum {j in JOBS} weight[j] * MachineForJob[j];
# jobs assigned to machine m stay within its capacity
subject to MachineCapacity {m in MACHINES}:
    numberof m in ({j in JOBS} MachineForJob[j]) <= capacity[m];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["JOBS"] = ["j1", "j2", "j3", "j4", "j5"]
ampl.param["weight"] = {"j1": 5, "j2": 4, "j3": 3, "j4": 2, "j5": 1}
ampl.param["number_of_machines"] = 3
ampl.param["capacity"] = {1: 2, 2: 2, 3: 1}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.var["MachineForJob"].to_dict())

HiGHS 1.14.0: optimal solution; objective 22
39 simplex iterations
1 branching nodes
{'j1': 1, 'j2': 1, 'j3': 2, 'j4': 2, 'j5': 3}


### Pure AMPL: data.dat and model.run

The two heaviest jobs fill machine 1, the next two machine 2, and the lightest job takes the one
place on machine 3. HiGHS has no counting constraints of its own; the MP interface rewrites
`numberof` with binary variables.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: JOBS: weight := j1 5  j2 4  j3 3  j4 2  j5 1;
param number_of_machines := 3;
param capacity := 1 2  2 2  3 1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display MachineForJob;

HiGHS 1.14.0: optimal solution; objective 22
39 simplex iterations
1 branching nodes
MachineForJob [*] :=
j1  1
j2  1
j3  2
j4  2
j5  3
;

